In [59]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh37')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe067.hpc.in.bmrc.ox.ac.uk:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [95]:
mt = hl.read_matrix_table('data/mt/ukb_wes_200k_annotated_chr21.mt')
#mt = qc.recalc_info(mt)
mt = mt.annotate_rows(consequence_category = annotation_case_builder(mt.vep.worst_csq_by_gene_canonical, mt.dbnsfp))


In [99]:
#mt.filter_rows(mt.consequence_category == 'ptv_LC')
mt = count_urv_by_samples(mt)
#mt.cols().flatten().show()


#variant_csqs_category_builder(mt.vep.worst_csq_for_variant_canonical, mt.dbnsfp)

In [92]:
def count_urv_by_samples(mt):
    r'''Count up ultra rare variants by cols and cateogry
    
    :param mt: a MatrixTable with the field "consequence_category"
    '''
    return mt.annotate_cols(n_coding_URV_SNP = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_coding_URV_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_URV_PTV = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                          n_URV_PTV_LC = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                          n_URV_damaging_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "damaging_missense")),
                          n_URV_other_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "other_missense")),
                          n_URV_synonymous = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "synonymous")),
                          n_URV_non_coding = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
                         )

In [102]:
# get prefixes to out files
prefix = "/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/all/211013_ptv"
burden = prefix + "/ukb_wes_200k_af02_chr20_burden.tsv.gz"
ko_prob = prefix + "/ukb_wes_200k_af02_chr14_ko_prob.tsv.gz"
kos = prefix + "/ukb_wes_200k_af02_chr15_knockouts.tsv.gz"

#mt = annotate_urv_cols(mt)
#mt = mt.annotate_rows(AAF_bin = analysis.maf_category_case_builder(mt.info))

hl.import_table(kos)

FatalError: HailException: Cannot load file 'file:/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/all/211013_ptv/ukb_wes_200k_af02_chr15_knockouts.tsv.gz'
  .gz cannot be loaded in parallel. Is the file actually *block* gzipped?
  If the file is actually block gzipped (even though its extension is .gz),
  use the 'force_bgz' argument to treat all .gz file extensions as .bgz.
  If you are sure that you want to load a non-block-gzipped file serially
  on one core, use the 'force' argument.

Java stack trace:
is.hail.utils.HailException: Cannot load file 'file:/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/all/211013_ptv/ukb_wes_200k_af02_chr15_knockouts.tsv.gz'
  .gz cannot be loaded in parallel. Is the file actually *block* gzipped?
  If the file is actually block gzipped (even though its extension is .gz),
  use the 'force_bgz' argument to treat all .gz file extensions as .bgz.
  If you are sure that you want to load a non-block-gz

2021-10-19 15:23:24 Hail: INFO: Coerced sorted dataset


In [33]:
mt.group_rows_by(mt.AAF_bin).aggregate(hl.agg.sum(mt.n_coding_URV_SNP))

TypeError: aggregate() takes 1 positional argument but 2 were given

2021-10-19 11:58:30 Hail: INFO: Coerced sorted dataset


In [64]:
PLOF_CSQS = ["transcript_ablation", "splice_acceptor_variant",
             "splice_donor_variant", "stop_gained", "frameshift_variant"]

MISSENSE_CSQS = ["stop_lost", "start_lost", "transcript_amplification",
                 "inframe_insertion", "inframe_deletion", "missense_variant"]

SYNONYMOUS_CSQS = ["stop_retained_variant", "synonymous_variant"]

OTHER_CSQS = ["mature_miRNA_variant", "5_prime_UTR_variant",
              "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant",
              "NMD_transcript_variant", "non_coding_transcript_variant", "upstream_gene_variant",
              "downstream_gene_variant", "TFBS_ablation", "TFBS_amplification", "TF_binding_site_variant",
              "regulatory_region_ablation", "regulatory_region_amplification", "feature_elongation",
              "regulatory_region_variant", "feature_truncation", "intergenic_variant"]


In [63]:
def variant_csqs_category_builder(csqs_vep_expr, csqs_dbnsfp_expr):
    r'''Create categories for downstream analysis'''
    return (hl.case()
            .when(hl.literal(set(PLOF_CSQS)).contains(csqs_stats_expr), "ptv")
            .when(hl.literal(set(MISSENSE_CSQS)).contains(csqs_stats_expr) & 
                   (~hl.is_defined(csqs_dbnsfp_expr.cadd_phred_score) | 
                    ~hl.is_defined(csqs_dbnsfp_expr.revel_score)), "other_missense")                                   
                .when(hl.literal(set(MISSENSE_CSQS)).contains(mt.vep.worst_csq_by_gene_canonical.most_severe_consequence) & 
                   (csqs_dbnsfp_expr.cadd_phred_score >= 20) & 
                   (csqs_dbnsfp_expr.revel_score >= 0.6), "damaging_missense") 
                .when(hl.literal(set(MISSENSE_CSQS)).contains(csqs_vep_expr), "other_missense")
                .when(hl.literal(set(SYNONYMOUS_CSQS)).contains(csqs_vep_expr), "synonymous")
                .when(hl.literal(set(OTHER_CSQS)).contains(csqs_vep_expr), "non_coding")
                .default("NA"))

In [78]:
def annotation_case_builder(worst_csq_by_gene_canonical_expr, csqs_dbnsfp_expr, use_loftee: bool = True):
    r'''Create categories for downstream analysis'''
    case = hl.case(missing_false=True)
    if use_loftee:
        case = (case
             .when(worst_csq_by_gene_canonical_expr.lof == 'HC', 'ptv')
             .when(worst_csq_by_gene_canonical_expr.lof == 'LC', 'ptv_LC')
            )
    else:
        case = case.when(hl.set(PLOF_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "ptv")
    case = (case.when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence) & 
                   (~hl.is_defined(csqs_dbnsfp_expr.cadd_phred_score) | ~hl.is_defined(csqs_dbnsfp_expr.revel_score)), "other_missense")                                   
                .when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence) & 
                   (csqs_dbnsfp_expr.cadd_phred_score >= 20) &  (csqs_dbnsfp_expr.revel_score >= 0.6), "damaging_missense") 
                .when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "other_missense")
                .when(hl.set(SYNONYMOUS_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "synonymous")
                .when(hl.set(OTHER_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "non_coding")
           )
    return case.or_missing()
                
    

In [73]:
def annotation_case_builder(worst_csq_by_gene_canonical_expr, use_loftee: bool = True, use_polyphen_and_sift: bool = False,
                            strict_definitions: bool = False):
    case = hl.case(missing_false=True)
    if use_loftee:
        case = (case
                .when(worst_csq_by_gene_canonical_expr.lof == 'HC', 'pLoF')
                .when(worst_csq_by_gene_canonical_expr.lof == 'LC', 'LC'))
    else:
        case = case.when(hl.set(PLOF_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'pLoF')
    if use_polyphen_and_sift:
        case = (case
                .when(missense.contains(mt.vep.worst_csq_for_variant_canonical.most_severe_consequence) &
                      (mt.vep.worst_csq_for_variant_canonical.polyphen_prediction == "probably_damaging") &
                      (mt.vep.worst_csq_for_variant_canonical.sift_prediction == "deleterious"), "damaging_missense")
                .when(missense.contains(mt.vep.worst_csq_for_variant_canonical.most_severe_consequence), "other_missense"))
    else:
        if strict_definitions:
            case = case.when(worst_csq_by_gene_canonical_expr.most_severe_consequence == 'missense_variant', 'missense')
        else:
            case = case.when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'missense')
    if strict_definitions:
        case = case.when(worst_csq_by_gene_canonical_expr.most_severe_consequence == 'synonymous_variant', 'synonymous')
    else:
        case = case.when(hl.set(SYNONYMOUS_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'synonymous')
    case = case.when(hl.set(OTHER_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'non-coding')
    return case.or_missing()